In [ ]:
import re
import polars as pl
from glob import glob

In [ ]:
glob('../redownload/**/*.xml')[0]

In [ ]:
with open('../redownload\\S01012002\\12002010442.xml', 'r', encoding='utf-8') as file:
    text = ''.join([i.strip() for i in file.readlines() if i.strip()!= ''])

In [ ]:
text

In [ ]:
article_attrib = re.search(r'<article (.+?)>', text).group(1)

article_attrib

In [ ]:
metadata = re.findall(r'(\w+?)=\"(.*?)\"', article_attrib)

metadata

In [ ]:
identifica = re.search(r'<Identifica><!\[CDATA\[(.+)\]\]></Identifica>', text)

identifica = identifica.group(1).strip() if identifica else identifica

identifica

In [ ]:
data = re.search(r'<Data><!\[CDATA\[(.+)\]\]></Data>', text)

data = data.group(1).strip() if data else data

data

In [ ]:
ementa = re.search(r'<Ementa>(<!\[CDATA\[)?(.+)(\]\]>)?</Ementa>', text)

ementa = ementa.group(2).strip() if ementa else ementa

ementa

In [ ]:
titulo = re.search(r'<Titulo>(<!\[CDATA\[)?(.+)(\]\]>)?</Titulo>', text)

titulo = titulo.group(2).strip() if titulo else titulo

titulo

In [ ]:
subtitulo = re.search(r'<SubTitulo><!\[CDATA\[(.+)\]\]></SubTitulo>', text)
subtitulo = subtitulo.group(1).strip() if subtitulo else subtitulo

subtitulo

In [ ]:
texto = re.search(r'<Texto>(<!\[CDATA\[)?(.+)(\]\]>)?</Texto>', text)
texto = texto.group(2).strip() if texto else texto
texto = ('\n'.join([re.sub(r'<\/?strong>', '', i) for i in re.findall(r'<p>(.+?)</p>', texto)])) if texto else texto

texto

In [ ]:
text

In [ ]:
autores = re.search(r'<Autores>(.+)</Autores>', text)
autores = re.findall(r'<assina>(.+?)</assina>', autores.group(1).strip()) if autores else autores

autores

In [ ]:
dir_tree = glob('../redownload/**/*.xml')
dir_tree = [i for i in dir_tree if int(i.split('\\')[1][-4:]) in range(2018, 2027)]

In [ ]:
df = pl.DataFrame()

for xml_path in dir_tree:
    with open(xml_path, 'r', encoding='utf-8') as file:
        text = ''.join([i.strip() for i in file.readlines() if i.strip()!= ''])
        frame = dict(
            re.findall(
                r'(\w+?)=\"(.*?)\"', 
                re.search(r'<article (.+?)>', text).group(1)
            )
        )
        
        identifica = re.search(r'<Identifica><!\[CDATA\[(.+)\]\]></Identifica>', text)
        frame['identifica'] = identifica.group(1).strip() if identifica else identifica
        
        data = re.search(r'<Data><!\[CDATA\[(.+)\]\]></Data>', text)
        frame['data'] = data.group(1).strip() if data else data
        
        ementa = re.search(r'<Ementa><!\[CDATA\[(.+)\]\]></Ementa>', text)
        frame['ementa'] = ementa.group(1).strip() if ementa else ementa
        
        titulo = re.search(r'<Titulo><!\[CDATA\[(.+)\]\]></Titulo>', text)
        frame['titulo'] = titulo.group(1).strip() if titulo else titulo
        
        subtitulo = re.search(r'<SubTitulo><!\[CDATA\[(.+)\]\]></SubTitulo>', text)
        frame['subtitulo'] = subtitulo.group(1).strip() if subtitulo else subtitulo
        
        texto = re.search(r'<Texto><!\[CDATA\[(.+)\]\]></Texto>', text)
        texto = texto.group(1).strip() if texto else texto
        frame['texto'] = ('\n'.join([re.sub(r'<\/?strong>', '', i) for i in re.findall(r'<p>(.+?)</p>', texto)])) if texto else texto
        
        autores = re.search(r'<Autores>(.+)</Autores>', text)
        frame['autores'] = str(re.findall(r'<assina>(.+?)</assina>', autores.group(1).strip())) if autores else autores
        
        df = pl.concat([df, pl.from_dict(frame)], how='diagonal_relaxed')


In [ ]:
df

In [ ]:
metadata = pl.read_parquet('../database_2/metadata*.parquet')
texto = pl.read_parquet('../database_2/texto*.parquet')
df = metadata.join(texto, on='id')

del metadata, texto

In [ ]:
df = (
    df
    .with_columns(
        pl.col('pubDate').str.to_date(format='%d/%m/%Y'),
        pl.col('artClass').str.strip_chars('a')
    )
    .filter(
        pl.col('pubDate').dt.year() >= 2018
    )
)

In [ ]:
df.describe()

In [ ]:
df.group_by('artClass', pl.col('pubName').str.head(3), 'artType').agg(pl.len()).sort(by='len', descending=True).filter(pl.col('len')>1).show(limit=20, fmt_str_lengths=100)

In [ ]:
df.filter(pl.col('artClass')=="00016:00011:00013:00008:00004:00000:00000:00000:00000:00000:00053:00001").select('texto').show(fmt_str_lengths=1000)

In [ ]:
df = (
    df
    .with_columns(
        vector=pl.col('artClass').map_elements(lambda x: list(map(lambda y: float(y)*1e-3, x.split(':'))), return_dtype=list[float])
    )
)

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import plotly.express as px

# Base para clusterização
base_cluster = df.select('id', 'pubName', 'artType', 'vector').to_pandas()

# Garante vetor numérico e remove linhas sem vetor válido
base_cluster['vector'] = base_cluster['vector'].apply(
    lambda v: np.asarray(v, dtype=np.float32) if isinstance(v, (list, tuple, np.ndarray)) else None
)
base_cluster = base_cluster[base_cluster['vector'].notna()].copy()

# Ajusta o shape: mantém somente vetores com o tamanho mais frequente
base_cluster['vector_len'] = base_cluster['vector'].apply(len)
dim_alvo = int(base_cluster['vector_len'].mode().iloc[0])
qtd_descartada = int((base_cluster['vector_len'] != dim_alvo).sum())

if qtd_descartada > 0:
    print(f'Removendo {qtd_descartada} documento(s) com dimensão diferente de {dim_alvo}.')

base_cluster = base_cluster[base_cluster['vector_len'] == dim_alvo].reset_index(drop=True)

if len(base_cluster) < 3:
    raise ValueError('Poucos documentos válidos para clusterização após limpeza dos vetores.')

X = np.vstack(base_cluster['vector'].to_numpy())

# Escolha automática do número de clusters pelo melhor silhouette
k_min = 2
k_max = min(12, len(base_cluster) - 1)

if k_max < k_min:
    raise ValueError('Poucos documentos para testar múltiplos clusters.')

avaliacoes = []
for k in range(k_min, k_max + 1):
    modelo = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = modelo.fit_predict(X)
    avaliacoes.append({
        'k': k,
        'inercia': modelo.inertia_,
        'silhouette': silhouette_score(X, labels)
    })

avaliacao_clusters = pd.DataFrame(avaliacoes)
melhor_k = int(avaliacao_clusters.loc[avaliacao_clusters['silhouette'].idxmax(), 'k'])
print(f'Melhor número de clusters (silhouette): {melhor_k}')

# Cluster final
kmeans = KMeans(n_clusters=melhor_k, random_state=42, n_init='auto')
base_cluster['cluster'] = kmeans.fit_predict(X).astype(str)

# PCA para visualização
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
base_cluster['pca_1'] = X_pca[:, 0]
base_cluster['pca_2'] = X_pca[:, 1]

avaliacao_clusters



In [ ]:
fig_metricas = px.line(
    avaliacao_clusters,
    x='k',
    y='silhouette',
    markers=True,
    title='Silhouette por número de clusters'
)
fig_metricas.show()

fig_clusters = px.scatter(
    base_cluster,
    x='pca_1',
    y='pca_2',
    color='cluster',
    hover_data=['id', 'pubName', 'artType'],
    opacity=0.75,
    title='Documentos agrupados por similaridade (vetor -> KMeans -> PCA)'
)
fig_clusters.show()

base_cluster[['id', 'cluster', 'pca_1', 'pca_2']].head()

